In [1]:
import json
import pandas as pd

def jprint(obj):
    print(json.dumps(obj, indent=2))


In [2]:
with open('../outputs/predictions-ragent-Qwen2.5-1.5B-Instruct-musique-grpo.jsonl') as f:
    records = [json.loads(line) for line in f]
print(len(records))

200


In [3]:
df = pd.DataFrame(records)
df.head()

,answer,n_hops,prompt,docs,answers,supporting_titles,trajectory
0,Casa Loma,3,[{'content': 'Answer the question based on the...,"[{'id': 4, 'is_supporting': False, 'text': '# ...",[Casa Loma],"[Sekou Lumumba, Casa Loma, Fall (Serena Ryder ...",[{'content': 'Answer the question based on the...
1,Yuan Shikai,3,[{'content': 'Answer the question based on the...,"[{'id': 0, 'is_supporting': False, 'text': '# ...",[Yuan Shikai],[Sino-Tibetan relations during the Ming dynast...,[{'content': 'Answer the question based on the...
2,Tualatin Mountains,3,[{'content': 'Answer the question based on the...,"[{'id': 5, 'is_supporting': True, 'text': '# P...",[Tualatin Mountains],"[Portland, Oregon, Coles Creek (Pennsylvania),...",[{'content': 'Answer the question based on the...
3,Casa Loma,3,[{'content': 'Answer the question based on the...,"[{'id': 1, 'is_supporting': False, 'text': '# ...",[Casa Loma],"[Arna Selznick, Casa Loma, Natalie Turner]",[{'content': 'Answer the question based on the...
4,1941,3,[{'content': 'Answer the question based on the...,"[{'id': 2, 'is_supporting': False, 'text': '# ...",[1941],"[Pacific War, Bang Bon District, Ercole Manfredi]",[{'content': 'Answer the question based on the...


In [4]:
from verifiers.rubrics.utils import get_last_answer

df['n_hops'] = df['supporting_titles'].apply(len)
df['predicted_answer'] = df['trajectory'].apply(get_last_answer)

[2025-03-13 02:15:06,766] [INFO] [real_accelerator.py:219:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/baris/miniconda3/envs/verifiers/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/baris/miniconda3/envs/verifiers/compiler_compat/ld: warning: libpthread.so.0, needed by /usr/local/cuda-12.3/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/baris/miniconda3/envs/verifiers/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda-12.3/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/baris/miniconda3/envs/verifiers/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda-12.3/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/baris/miniconda3/envs/verifiers/compiler_compat/ld: /usr/local/cuda-12.3/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/home/baris/miniconda3/envs/verifiers/compiler_compat/ld: /usr/local/cuda-12.3/lib64/libcufile.so: undefined reference to `__gxx_personality_v0@CXXABI_

INFO 03-13 02:15:08 __init__.py:183] Automatically detected platform cuda.


In [5]:
from verifiers.metrics.musique import compute_metrics, exact_match, f1

df['exact_match'] = df.apply(lambda row: exact_match(row['predicted_answer'], row['answers']), axis=1)
df['f1'] = df.apply(lambda row: f1(row['predicted_answer'], row['answers']), axis=1)

df.head()

,answer,n_hops,prompt,docs,answers,supporting_titles,trajectory,predicted_answer,exact_match,f1
0,Casa Loma,3,[{'content': 'Answer the question based on the...,"[{'id': 4, 'is_supporting': False, 'text': '# ...",[Casa Loma],"[Sekou Lumumba, Casa Loma, Fall (Serena Ryder ...",[{'content': 'Answer the question based on the...,Toronto,0,0.0
1,Yuan Shikai,3,[{'content': 'Answer the question based on the...,"[{'id': 0, 'is_supporting': False, 'text': '# ...",[Yuan Shikai],[Sino-Tibetan relations during the Ming dynast...,[{'content': 'Answer the question based on the...,Yuan Shikai,1,1.0
2,Tualatin Mountains,3,[{'content': 'Answer the question based on the...,"[{'id': 5, 'is_supporting': True, 'text': '# P...",[Tualatin Mountains],"[Portland, Oregon, Coles Creek (Pennsylvania),...",[{'content': 'Answer the question based on the...,Tualatin Mountains,1,1.0
3,Casa Loma,3,[{'content': 'Answer the question based on the...,"[{'id': 1, 'is_supporting': False, 'text': '# ...",[Casa Loma],"[Arna Selznick, Casa Loma, Natalie Turner]",[{'content': 'Answer the question based on the...,Marksburg Castle,0,0.0
4,1941,3,[{'content': 'Answer the question based on the...,"[{'id': 2, 'is_supporting': False, 'text': '# ...",[1941],"[Pacific War, Bang Bon District, Ercole Manfredi]",[{'content': 'Answer the question based on the...,1941,1,1.0


In [7]:
df.groupby('n_hops')[['exact_match', 'f1']].agg(['count', 'mean', 'std'])

exact_match                    f1                    
             count  mean       std count      mean       std
n_hops                                                      
3              100  0.12  0.326599   100  0.164002  0.338665
4              100  0.07  0.256432   100  0.132833  0.304102